# 02b — Training HRNet via MMPose (W18 / W32 / W48)

**Runtime:** GPU V100 direkomendasikan. T4 bisa untuk W18/W32, kurang ideal untuk W48.

**Prereq:** `01_dataset_conversion.ipynb` sudah dijalankan dan COCO JSON sudah terverifikasi.

**Estimasi waktu:** W18 ~8h, W32 ~12h, W48 ~18h di V100.

In [ ]:
# ── Cell 1: Install MMPose ────────────────────────────────────────────────────
# Urutan install sangat penting — jangan diubah
!pip install torch==2.1.0 torchvision==0.16.0 --quiet
!pip install -U openmim --quiet
!mim install mmengine --quiet
!mim install 'mmcv>=2.0.0' --quiet
!mim install mmdet --quiet
!mim install mmpose --quiet
print('✅ MMPose stack installed')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/USERNAME/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst kalau src ada dan dst belum ada. Aman dipanggil berkali-kali."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')
    # Output notebook 01 (COCO JSON) — attach "01-dataset-conversion" via "+ Add Input" -> Notebook Output
    _link(f'/kaggle/input/01-dataset-conversion/soccer-homography-benchmark/data/converted/annotations',
          f'{PROJECT_ROOT}/data/converted/annotations')
    # Symlink gambar asli ke struktur data/converted/images/{split} (dibutuhkan training MMPose)
    for _split in ['train', 'val', 'test']:
        _link(f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/{_split}/images',
              f'{PROJECT_ROOT}/data/converted/images/{_split}')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

COCO_ANN_DIR = f'{PROJECT_ROOT}/data/converted/annotations'
IMAGES_DIR   = f'{PROJECT_ROOT}/data/converted/images'
for split in ['train', 'val', 'test']:
    p = f'{COCO_ANN_DIR}/{split}.json'
    print(f'  {split}.json: {"OK" if os.path.exists(p) else "TIDAK ADA — jalankan 01_dataset_conversion dulu!"}')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Cell 3: Verifikasi MMPose install ────────────────────────────────────────
import mmpose
import torch
print(f'MMPose version : {mmpose.__version__}')
print(f'PyTorch        : {torch.__version__}')
print(f'CUDA           : {torch.version.cuda}')
print(f'GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')


In [ ]:
# ── Cell 4: Build MMPose config untuk HRNet ──────────────────────────────────
# MMPose menggunakan .py config files, bukan YAML.
# Kita buat config dict yang dipass ke API training.

import copy
from mmengine.config import Config
from mim.commands.download import download as mim_download

# Download base config dari MMPose
!mim download mmpose --config td-hm_hrnet-w18_8xb64-210e_coco-256x192 --dest /tmp/mmpose_configs/

print('✅ Base config downloaded')

In [ ]:
# ── Cell 5: Buat custom config untuk dataset soccer field ─────────────────────
import os

def make_hrnet_config(variant='w18', coco_ann_dir=COCO_ANN_DIR, images_dir=IMAGES_DIR, output_dir=None):
    """
    Generate MMPose config dict untuk HRNet pada dataset soccer field.
    Modifikasi utama dari config COCO standar:
      - num_keypoints: 17 → 32
      - dataset path → custom COCO JSON
      - output dir → artifacts/
    """
    base_config_map = {
        'w18': '/tmp/mmpose_configs/td-hm_hrnet-w18_8xb64-210e_coco-256x192.py',
        'w32': '/tmp/mmpose_configs/td-hm_hrnet-w32_8xb64-210e_coco-256x192.py',
        'w48': '/tmp/mmpose_configs/td-hm_hrnet-w48_8xb64-210e_coco-256x192.py',
    }
    
    cfg = Config.fromfile(base_config_map[variant])
    
    # Ubah jumlah keypoint: 17 → 32
    cfg.model.head.num_joints = 32
    cfg.model.head.loss.num_joints = 32
    
    # Dataset paths
    for split in ['train_dataloader', 'val_dataloader', 'test_dataloader']:
        split_name = split.replace('_dataloader', '').replace('train', 'train')
        cfg[split].dataset.ann_file = f'{coco_ann_dir}/{split_name}.json'
        cfg[split].dataset.data_prefix = dict(img=images_dir)
        cfg[split].dataset.metainfo = dict(num_keypoints=32)
    
    # Output dir
    if output_dir:
        cfg.work_dir = output_dir
    
    # Epochs untuk benchmark yang lebih pendek
    cfg.train_cfg.max_epochs = 300
    
    return cfg

cfg_w18 = make_hrnet_config('w18', output_dir=f'{PROJECT_ROOT}/artifacts/weights/hrnet/w18')
print('✅ HRNet-W18 config ready')

In [ ]:
# ── Cell 6: Training HRNet — satu variant ────────────────────────────────────
# Jalankan cell ini TIGA KALI, ganti variant di bawah setiap kali.
# Restart GPU memory antar run dengan: Runtime → Restart runtime

VARIANT = 'w18'  # Ganti ke 'w32' atau 'w48' untuk variant berikutnya

from mmengine.runner import Runner
import gc, torch

output_dir = f'{PROJECT_ROOT}/artifacts/weights/hrnet/{VARIANT}'
os.makedirs(output_dir, exist_ok=True)

cfg = make_hrnet_config(VARIANT, output_dir=output_dir)

t_start = datetime.now()
runner = Runner.from_cfg(cfg)
runner.train()
t_end = datetime.now()

duration = (t_end - t_start).total_seconds() / 3600

log = {
    'model_family'  : 'hrnet',
    'model_variant' : VARIANT,
    'trained_at'    : t_start.isoformat(),
    'duration_hours': round(duration, 2),
    'gpu'           : torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}
with open(f'{output_dir}/train_log.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'\n✅ HRNet-{VARIANT.upper()} selesai! Duration: {duration:.1f}h')

# Bersihkan memory
del runner
gc.collect()
torch.cuda.empty_cache()
print('🧹 GPU memory cleared — siap untuk variant berikutnya')